# Mundiales 2018, 2022 y 2026
## Preparación de datos y entrada al análisis supervisado

Trabajaremos exclusivamente con la fase de grupos. Los archivos contienen errores deliberados. No uses la base del profesor.

## Objetivos

- Perfilar datos.
- Estandarizar esquemas.
- Limpiar fechas, equipos, goles y marcadores.
- Eliminar duplicados con criterio.
- Comparar torneos mediante tasas.
- Construir variables conocidas antes de cada partido.
- Entrenar un primer clasificador y detectar fuga de información.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

DATA = Path('../datos')
pd.set_option('display.max_columns', 50)

## Cargar los tres archivos

In [ ]:
d18 = pd.read_csv('mundial_2018_sucio.csv', dtype=str)
d22 = pd.read_csv('mundial_2022_sucio.csv', dtype=str)
d26 = pd.read_csv('mundial_2026_sucio.csv', dtype=str)

for nombre, df in [('2018', d18), ('2022', d22), ('2026', d26)]:
    print(f"=== {nombre} ===")
    print("Dimensiones:", df.shape)
    display(df.head())
    print()

## Perfilado

Para cada archivo revisa:

- columnas;
- tipos;
- valores nulos;
- duplicados;
- valores únicos de grupos, fases y equipos;
- goles que no puedan convertirse directamente a número.

In [ ]:
def perfil(df, nombre):
    print(f"=== {nombre} ===")
    print("Filas:", len(df), "Columnas:", list(df.columns))
    print("\nNulos por columna:")
    print(df.isna().sum())
    print("\nDuplicados exactos:", df.duplicated().sum())
    print("\nValores únicos de columnas categóricas relevantes:")
    for col in df.columns:
        if df[col].nunique() < 20:
            print(f"  {col}: {sorted(df[col].dropna().unique().tolist())}")

perfil(d18, '2018')
perfil(d22, '2022')
perfil(d26, '2026')

## Unificar nombres de columnas

In [ ]:
rename_maps = {
    2018: {
        'ID Partido': 'partido_id', 'Año': 'mundial', 'Fase': 'fase',
        'Grupo': 'grupo', 'Jornada': 'jornada', 'Fecha': 'fecha',
        'Equipo Local': 'equipo_local', 'Equipo Visitante': 'equipo_visitante',
        'Goles Local': 'goles_local', 'Goles Visitante': 'goles_visitante',
        'Marcador': 'marcador', 'Anfitrión Local': 'local_es_anfitrion',
        'Fuente': 'fuente',
    },
    2022: {
        'match_id': 'partido_id', 'WorldCup': 'mundial', 'stage': 'fase',
        'group_name': 'grupo', 'match_day': 'jornada', 'date': 'fecha',
        'local': 'equipo_local', 'visitor': 'equipo_visitante',
        'home_score': 'goles_local', 'away_score': 'goles_visitante',
        'score_text': 'marcador', 'home_host': 'local_es_anfitrion',
        'source_url': 'fuente',
    },
    2026: {
        'match': 'partido_id', 'wc': 'mundial', 'round': 'fase',
        'grp': 'grupo', 'md': 'jornada', 'played_on': 'fecha',
        'home': 'equipo_local', 'away': 'equipo_visitante',
        'HG': 'goles_local', 'AG': 'goles_visitante',
        'result_raw': 'marcador', 'host_h': 'local_es_anfitrion',
        'host_a': 'visitante_es_anfitrion', 'source': 'fuente',
    },
}

columnas_base = [
    'partido_id', 'mundial', 'fase', 'grupo', 'jornada', 'fecha',
    'equipo_local', 'equipo_visitante', 'goles_local',
    'goles_visitante', 'marcador', 'local_es_anfitrion',
    'visitante_es_anfitrion', 'fuente'
]

## Normalizar equipos

No conviene borrar acentos del nombre que se mostrará. Crea una clave auxiliar sin acentos, minúscula y sin signos para buscar en el catálogo.

In [ ]:
catalogo = pd.read_csv('catalogo_equipos.csv')

def clave_texto(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'[^a-z0-9]+', ' ', texto).strip()
    return texto

catalogo['clave'] = catalogo['variante'].apply(clave_texto)
mapa_equipos = dict(zip(catalogo['clave'], catalogo['nombre_canonico']))

def equipo_canonico(valor):
    clave = clave_texto(valor)
    return mapa_equipos.get(clave, valor)


## Fechas, grupos, booleanos y marcadores

In [ ]:
rangos = {
    2018: ('2018-06-14', '2018-06-28'),
    2022: ('2022-11-20', '2022-12-02'),
    2026: ('2026-06-11', '2026-06-27'),
}

def convertir_fecha(valor, mundial):
    if pd.isna(valor):
        return pd.NaT
    texto = str(valor).strip()
    # serial de Excel (número puro)
    if re.fullmatch(r'\d{4,6}', texto):
        try:
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(texto))
        except Exception:
            pass
    formatos = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%d-%m-%y',
                '%b %d, %Y', '%d/%m/%y', '%Y/%m/%d']
    ini, fin = pd.Timestamp(rangos[mundial][0]), pd.Timestamp(rangos[mundial][1])
    for fmt in formatos:
        try:
            f = pd.to_datetime(texto, format=fmt)
            if ini <= f <= fin:
                return f
        except Exception:
            continue
    # último intento: parseo flexible de pandas
    try:
        f = pd.to_datetime(texto, dayfirst=True, errors='coerce')
        if pd.notna(f) and ini <= f <= fin:
            return f
    except Exception:
        pass
    return pd.NaT

def extraer_numero(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor)
    m = re.search(r'-?\d+', texto)
    if m:
        n = int(m.group())
        return n if n >= 0 else np.nan
    return np.nan

def separar_marcador(valor):
    if pd.isna(valor):
        return np.nan, np.nan
    texto = str(valor).strip()
    texto = re.sub(r'[–—x X:]', '-', texto)
    texto = re.sub(r'-+', '-', texto)
    partes = texto.split('-')
    partes = [p.strip() for p in partes if p.strip() != '']
    if len(partes) == 2 and all(p.isdigit() for p in partes):
        return int(partes[0]), int(partes[1])
    return np.nan, np.nan

def normalizar_grupo(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().upper()
    if texto in ('S/D', 'N/D', 'ND', ''):
        return np.nan
    texto_sin_prefijo = re.sub(r'\b(GRUPO|GROUP)\b', ' ', texto)
    m = re.search(r'\b([A-L])\b', texto_sin_prefijo)
    return m.group(1) if m else np.nan

def normalizar_booleano(valor):
    if pd.isna(valor):
        return False
    texto = str(valor).strip().lower()
    return texto in ('si', 'sí', '1', 'true', 'verdadero')

## Función de limpieza reproducible

In [ ]:
sedes = {
    2018: {'Russia'},
    2022: {'Qatar'},
    2026: {'United States', 'Mexico', 'Canada'},
}

def limpiar_mundial(df, mundial):
    df = df.rename(columns=rename_maps[mundial]).copy()

    for col in columnas_base:
        if col not in df.columns:
            df[col] = np.nan

    df = df[columnas_base].copy()

    df['mundial'] = mundial
    df['fase'] = 'Fase de grupos'
    df['equipo_local'] = df['equipo_local'].apply(equipo_canonico)
    df['equipo_visitante'] = df['equipo_visitante'].apply(equipo_canonico)
    df['grupo'] = df['grupo'].apply(normalizar_grupo)
    df['jornada'] = df['jornada'].apply(extraer_numero)
    df['fecha'] = df['fecha'].apply(lambda v: convertir_fecha(v, mundial))
    df['local_es_anfitrion'] = df['local_es_anfitrion'].apply(normalizar_booleano) | df['equipo_local'].isin(sedes[mundial])
    df['visitante_es_anfitrion'] = df['visitante_es_anfitrion'].apply(normalizar_booleano) | df['equipo_visitante'].isin(sedes[mundial])

    goles_l = df['goles_local'].apply(extraer_numero)
    goles_v = df['goles_visitante'].apply(extraer_numero)
    marc_l, marc_v = zip(*df['marcador'].apply(separar_marcador))
    marc_l, marc_v = pd.Series(marc_l, index=df.index), pd.Series(marc_v, index=df.index)

    # Regla: si el marcador se puede leer, prevalece; si no, usar goles sueltos
    df['goles_local'] = marc_l.combine_first(goles_l)
    df['goles_visitante'] = marc_v.combine_first(goles_v)

    df['marcador'] = df.apply(
        lambda r: f"{int(r['goles_local'])}-{int(r['goles_visitante'])}"
        if pd.notna(r['goles_local']) and pd.notna(r['goles_visitante']) else np.nan,
        axis=1)

    # Grupos faltantes: inferir por el grupo más frecuente que ya tenga ese equipo en ese mundial
    mapa_grupo = (df.dropna(subset=['grupo'])
                    .groupby('equipo_local')['grupo'].agg(lambda s: s.value_counts().idxmax())
                    .to_dict())
    faltantes = df['grupo'].isna()
    df.loc[faltantes, 'grupo'] = df.loc[faltantes, 'equipo_local'].map(mapa_grupo)

    df = df.drop_duplicates(subset=['partido_id'], keep='first')

    df['resultado_local'] = np.select(
        [df['goles_local'] > df['goles_visitante'],
         df['goles_local'] == df['goles_visitante']],
        ['Gana', 'Empata'], default='Pierde')
    df['goles_totales'] = df['goles_local'] + df['goles_visitante']
    df['diferencia_goles'] = df['goles_local'] - df['goles_visitante']

    return df

limpio18 = limpiar_mundial(d18, 2018)
limpio22 = limpiar_mundial(d22, 2022)
limpio26 = limpiar_mundial(d26, 2026)
partidos = pd.concat([limpio18, limpio22, limpio26], ignore_index=True)

## Validaciones obligatorias

La limpieza no termina cuando el código deja de producir errores. Debes comprobar invariantes.

In [ ]:
conteo = partidos['mundial'].value_counts()
assert conteo.get(2018) == 48, f"2018 tiene {conteo.get(2018)} partidos, se esperaban 48"
assert conteo.get(2022) == 48, f"2022 tiene {conteo.get(2022)} partidos, se esperaban 48"
assert conteo.get(2026) == 72, f"2026 tiene {conteo.get(2026)} partidos, se esperaban 72"

assert partidos['partido_id'].duplicated().sum() == 0, "Hay duplicados de partido_id"
assert (partidos[['goles_local', 'goles_visitante']] < 0).any().any() == False, "Hay goles negativos"
assert partidos[['equipo_local', 'equipo_visitante', 'goles_local',
                  'goles_visitante', 'grupo']].isna().sum().sum() == 0, "Hay nulos en campos críticos"

marcador_calc = partidos['goles_local'].astype(int).astype(str) + '-' + partidos['goles_visitante'].astype(int).astype(str)
assert (marcador_calc == partidos['marcador']).all(), "Marcador inconsistente con los goles"

conteo_grupos = partidos.groupby(['mundial', 'grupo']).size()
assert (conteo_grupos == 6).all(), f"Hay grupos con partidos distintos de 6:\n{conteo_grupos[conteo_grupos != 6]}"

grupos_validos = set('ABCDEFGHIJKL')
assert partidos['grupo'].dropna().isin(grupos_validos).all(), "Hay valores de grupo fuera de A-L"
print("Todas las validaciones pasaron. Filas totales:", len(partidos))

## Comparación de los Mundiales

In [ ]:
comparacion = (partidos.groupby('mundial')
    .agg(partidos_totales=('partido_id', 'count'),
         goles=('goles_totales', 'sum'),
         empates=('resultado_local', lambda s: (s == 'Empata').sum()))
    .reset_index())
comparacion['goles_por_partido'] = comparacion['goles'] / comparacion['partidos_totales']
comparacion['pct_empates'] = comparacion['empates'] / comparacion['partidos_totales']
display(comparacion)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(comparacion['mundial'].astype(str), comparacion['goles_por_partido'], color='#4C72B0')
ax.set_ylabel('Goles por partido')
ax.set_title('Goles por partido por Mundial (tasa, no total)')
plt.show()

## Tabla por equipo

In [ ]:
locales = partidos.rename(columns={
    'equipo_local': 'equipo', 'equipo_visitante': 'rival',
    'goles_local': 'gf', 'goles_visitante': 'gc'})[
    ['mundial', 'equipo', 'rival', 'gf', 'gc', 'resultado_local']]
locales['puntos'] = locales['resultado_local'].map({'Gana': 3, 'Empata': 1, 'Pierde': 0})

visitantes = partidos.rename(columns={
    'equipo_visitante': 'equipo', 'equipo_local': 'rival',
    'goles_visitante': 'gf', 'goles_local': 'gc'})[
    ['mundial', 'equipo', 'rival', 'gf', 'gc', 'resultado_local']]
visitantes['resultado_visita'] = visitantes['resultado_local'].map(
    {'Gana': 'Pierde', 'Empata': 'Empata', 'Pierde': 'Gana'})
visitantes['puntos'] = visitantes['resultado_visita'].map({'Gana': 3, 'Empata': 1, 'Pierde': 0})
visitantes = visitantes.drop(columns=['resultado_local']).rename(columns={'resultado_visita': 'resultado_local'})

apariciones = pd.concat([locales, visitantes], ignore_index=True)

tabla_equipos = (apariciones.groupby(['mundial', 'equipo'])
    .agg(PJ=('equipo', 'count'),
         PG=('resultado_local', lambda s: (s == 'Gana').sum()),
         PE=('resultado_local', lambda s: (s == 'Empata').sum()),
         PP=('resultado_local', lambda s: (s == 'Pierde').sum()),
         GF=('gf', 'sum'), GC=('gc', 'sum'), PTS=('puntos', 'sum'))
    .reset_index())
tabla_equipos['DG'] = tabla_equipos['GF'] - tabla_equipos['GC']
tabla_equipos['PTS_por_partido'] = tabla_equipos['PTS'] / tabla_equipos['PJ']
display(tabla_equipos.sort_values(['mundial', 'PTS'], ascending=[True, False]))

## Variables previas al partido

Para predecir no podemos utilizar datos ocurridos después del inicio. Crearemos promedios acumulados antes de cada encuentro.

In [ ]:
def construir_variables_previas(partidos):
    partidos = partidos.sort_values(['mundial', 'jornada', 'partido_id']).reset_index(drop=True)
    stats = {}  # (mundial, equipo) -> dict con PJ, puntos, GF, GC acumulados
    filas = []

    for _, r in partidos.iterrows():
        key_l = (r['mundial'], r['equipo_local'])
        key_v = (r['mundial'], r['equipo_visitante'])
        sl = stats.get(key_l, {'PJ': 0, 'PTS': 0, 'GF': 0, 'GC': 0})
        sv = stats.get(key_v, {'PJ': 0, 'PTS': 0, 'GF': 0, 'GC': 0})

        fila = r.to_dict()
        fila['local_pts_prom_pre'] = sl['PTS'] / sl['PJ'] if sl['PJ'] else 0
        fila['visita_pts_prom_pre'] = sv['PTS'] / sv['PJ'] if sv['PJ'] else 0
        fila['local_gd_prom_pre'] = (sl['GF'] - sl['GC']) / sl['PJ'] if sl['PJ'] else 0
        fila['visita_gd_prom_pre'] = (sv['GF'] - sv['GC']) / sv['PJ'] if sv['PJ'] else 0
        fila['local_gf_prom_pre'] = sl['GF'] / sl['PJ'] if sl['PJ'] else 0
        fila['visita_gf_prom_pre'] = sv['GF'] / sv['PJ'] if sv['PJ'] else 0
        filas.append(fila)

        pts_l = 3 if r['resultado_local'] == 'Gana' else (1 if r['resultado_local'] == 'Empata' else 0)
        pts_v = 3 if r['resultado_local'] == 'Pierde' else (1 if r['resultado_local'] == 'Empata' else 0)
        stats[key_l] = {'PJ': sl['PJ'] + 1, 'PTS': sl['PTS'] + pts_l,
                         'GF': sl['GF'] + r['goles_local'], 'GC': sl['GC'] + r['goles_visitante']}
        stats[key_v] = {'PJ': sv['PJ'] + 1, 'PTS': sv['PTS'] + pts_v,
                         'GF': sv['GF'] + r['goles_visitante'], 'GC': sv['GC'] + r['goles_local']}

    return pd.DataFrame(filas)

features_df = construir_variables_previas(partidos)

## Entrenamiento y prueba

In [ ]:
features = [
    'jornada',
    'local_pts_prom_pre', 'visita_pts_prom_pre',
    'local_gd_prom_pre', 'visita_gd_prom_pre',
    'local_gf_prom_pre', 'visita_gf_prom_pre',
    'local_es_anfitrion', 'visitante_es_anfitrion'
]

train = features_df[features_df['mundial'].isin([2018, 2022])].copy()
test = features_df[features_df['mundial'] == 2026].copy()

X_train, y_train = train[features], train['resultado_local']
X_test, y_test = test[features], test['resultado_local']

# línea base: siempre predecir la clase más frecuente en entrenamiento
clase_base = y_train.value_counts().idxmax()
baseline_preds = [clase_base] * len(y_test)
print("Línea base (clase mayoritaria):", clase_base)
print("Accuracy línea base:", accuracy_score(y_test, baseline_preds))

modelo = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
modelo.fit(X_train, y_train)
preds = modelo.predict(X_test)
print("Accuracy árbol:", accuracy_score(y_test, preds))

ConfusionMatrixDisplay.from_predictions(y_test, preds)
plt.show()

plt.figure(figsize=(16, 8))
plot_tree(modelo, feature_names=features, class_names=modelo.classes_, filled=True, fontsize=8)
plt.show()

## Experimento de fuga de información

Agrega temporalmente `goles_local`, `goles_visitante` y `diferencia_goles` como variables. Si la precisión sube de forma extrema, explica por qué el modelo no está prediciendo realmente.

In [ ]:
features_fuga = features + ['goles_local', 'goles_visitante', 'diferencia_goles']
X_train_f, X_test_f = train[features_fuga], test[features_fuga]

modelo_fuga = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
modelo_fuga.fit(X_train_f, y_train)
preds_fuga = modelo_fuga.predict(X_test_f)
print("Accuracy con fuga de información:", accuracy_score(y_test, preds_fuga))

## Reflexión final

Responde:

- ¿Qué problema de calidad fue el más difícil?
  La extracción del grupo (A–L) fue la más delicada: una extracción ingenua de "la primera letra A-L" confundía la G de "Grupo" con el grupo real, y la D del centinela "s/d" con el Grupo D. Hubo que exigir que la letra apareciera como palabra completa y tratar "s/d"/"N/D" como vacío antes de inferir el grupo desde el historial del equipo.

- ¿Qué decisión de limpieza podría cambiar los resultados?

  La regla de "el marcador prevalece sobre los goles sueltos cuando ambos existen" resolvió 3 conflictos reales (uno por torneo, ej. M-2018-23 con goles -1/3 pero marcador "0:3"). Si se hubiera usado la regla inversa, esos 3 partidos habrían quedado con goles negativos o inconsistentes, afectando la tabla de posiciones y las variables previas del modelo.

- ¿Por qué 2026 debe compararse mediante tasas?
  2026 tiene 72 partidos contra 48 de 2018 y 2022 (50% más), así que sus totales (215 goles vs 122 y 120) son mayores solo por tener más partidos. En goles por partido la diferencia es mucho menor: 2.99 (2026) vs 2.54 (2018) vs 2.50 (2022). Comparar totales sin ajustar por número de partidos exagera la diferencia real.

- ¿El árbol supera la línea base?
  Sí. La línea base (predecir siempre "Pierde", la clase más frecuente en entrenamiento) da un accuracy de 0.25. El árbol de decisión alcanza 0.36. Es una mejora real pero modesta: el modelo capta algo de señal, pero está lejos de ser confiable con tan pocos datos de entrenamiento (96 partidos).

- ¿Qué variables reales agregarías para mejorar una predicción?
  Ranking FIFA antes del torneo, valor de mercado o edad promedio del plantel, resultados en clasificatorias, lesiones de jugadores clave, y distancia/aclimatación (relevante en 2026 por las tres sedes).

- ¿Por qué un resultado de 100 % puede ser una señal de alarma?
  En el experimento de fuga, agregar goles_local, goles_visitante y diferencia_goles como variables predictoras subió el accuracy a 1.0. Esto no significa que el modelo "aprendió a predecir": esas variables ya contienen el resultado del partido (diferencia_goles > 0 implica "Gana" por definición), así que el árbol solo está leyendo la respuesta, no prediciéndola. Un 100% de precisión debe hacer sospechar fuga de información antes que celebrarse.
